In [ ]:
from pathlib import Path
import json
import hashlib
import os
import subprocess
import sys
import time

PROJECT = Path("/Users/heqiuyan/Desktop/biohub-cell-tracking")

CKPT = (
    PROJECT
    / "models"
    / "official_baseline"
    / "official_baseline_fold0_best.pth"
)

TRAIN_DIR = (
    PROJECT
    / "data"
    / "raw"
    / "competition"
    / "biohub-cell-tracking-during-development"
    / "train"
)

SPLITS = PROJECT / "configs" / "dataset_splits.json"
PREDICT_SCRIPT = PROJECT / "scripts" / "predict_unet_transformer_mps.py"

print("Project:", PROJECT)
print("Checkpoint:", CKPT.exists())
print("Train data:", TRAIN_DIR.exists())
print("Splits:", SPLITS.exists())
print("Predict script:", PREDICT_SCRIPT.exists())

# Verify checkpoint identity
sha = hashlib.sha256(CKPT.read_bytes()).hexdigest()
print("\nCheckpoint SHA256:")
print(sha)

EXPECTED_SHA = "4a3943a2a4d7a8e288851b5736972d84793e29f88fe334c124597da288fbb600"
assert sha == EXPECTED_SHA

print("\nPreflight: OK")

In [ ]:
import json

CONFIG_PATH = CKPT.parent / "config.json"

config = {
    "unet_out_channels": 32,
    "unet_layers": [32, 64, 128],
    "downsample": [1, 4, 4],
    "window_size": 2,
}

CONFIG_PATH.write_text(json.dumps(config, indent=2))

print("Config path:", CONFIG_PATH)
print(CONFIG_PATH.read_text())

assert CONFIG_PATH.exists()
print("\nConfig: OK")

In [ ]:
import os
import sys
import time
import subprocess
from pathlib import Path

env = os.environ.copy()

env["PYTHONPATH"] = ":".join([
    str(PROJECT / "scripts"),
    str(PROJECT / "external" / "official" / "scripts"),
    str(PROJECT / "external" / "official" / "src"),
    str(PROJECT / "src"),
])

# 固定输出目录用户名，方便后面找到 predictions
env["USER"] = "markdd520"

# MPS 遇到个别不支持算子时允许 CPU fallback
env["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

cmd = [
    sys.executable,
    str(PREDICT_SCRIPT),
    "--data-dir", str(TRAIN_DIR),
    "--splits", str(SPLITS),
    "--split", "0",
    "--method", "official_baseline_fold0",
    "--weights", str(CKPT),
    "--det-threshold", "0.99",
    "--slice", ":1",
]

print("Python:", sys.executable)
print("Running:")
print(" ".join(cmd))
print()

start = time.perf_counter()

process = subprocess.Popen(
    cmd,
    cwd=PROJECT,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

returncode = process.wait()
elapsed = time.perf_counter() - start

print(f"\nReturn code: {returncode}")
print(f"Wall time: {elapsed:.1f} s = {elapsed / 60:.2f} min")

assert returncode == 0, f"Inference failed with return code {returncode}"

print("\nSmoke inference: OK")

In [ ]:
import tracksdata as td

PRED_DIR = (
    PROJECT
    / "predictions"
    / "markdd520"
    / "official_baseline_fold0"
    / "split_0"
)

pred_files = sorted(PRED_DIR.glob("*.geff"))

print("Prediction files:", len(pred_files))

for p in pred_files:
    loaded = td.graph.IndexedRXGraph.from_geff(p)

    # tracksdata 当前版本可能返回 (graph, metadata)
    graph = loaded[0] if isinstance(loaded, tuple) else loaded

    print("\nDataset:", p.stem)
    print("Nodes:", graph.num_nodes())
    print("Edges:", graph.num_edges())
    print("Path:", p)

In [ ]:
import pandas as pd

CV_PATH = PROJECT / "configs" / "cv_manifest.csv"
cv = pd.read_csv(CV_PATH)

name = "44b6_18ced818"

row = cv.loc[cv["dataset"] == name].iloc[0]

pred_path = PRED_DIR / f"{name}.geff"
loaded = td.graph.IndexedRXGraph.from_geff(pred_path)
graph = loaded[0] if isinstance(loaded, tuple) else loaded

pred_nodes = graph.num_nodes()
pred_edges = graph.num_edges()

estimated_nodes = int(row["estimated_nodes"])
sparse_gt_nodes = int(row["sparse_gt_nodes"])
n_frames = int(row["n_frames"])

print("Dataset:", name)
print("Frames:", n_frames)

print("\nEstimated total nodes:", estimated_nodes)
print("Estimated / frame:", estimated_nodes / n_frames)

print("\nSparse GT nodes:", sparse_gt_nodes)
print("Sparse GT / frame:", sparse_gt_nodes / n_frames)

print("\nPredicted nodes:", pred_nodes)
print("Predicted / frame:", pred_nodes / n_frames)

print("\nPred / estimated ratio:", pred_nodes / estimated_nodes)
print("Predicted edges:", pred_edges)

In [ ]:
import os
import sys
import time
import subprocess

env = os.environ.copy()

env["PYTHONPATH"] = ":".join([
    str(PROJECT / "scripts"),
    str(PROJECT / "external" / "official" / "scripts"),
    str(PROJECT / "external" / "official" / "src"),
    str(PROJECT / "src"),
])

env["USER"] = "markdd520"
env["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

cmd = [
    sys.executable,
    str(PREDICT_SCRIPT),
    "--data-dir", str(TRAIN_DIR),
    "--splits", str(SPLITS),
    "--split", "0",
    "--method", "official_baseline_fold0",
    "--weights", str(CKPT),
    "--det-threshold", "0.99",
]

print("Running FULL Fold 0 inference")
print("Expected datasets: 40")
print("Device: MPS")
print()

start = time.perf_counter()

process = subprocess.Popen(
    cmd,
    cwd=PROJECT,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

returncode = process.wait()
elapsed = time.perf_counter() - start

print(f"\nReturn code: {returncode}")
print(f"Wall time: {elapsed:.1f} s = {elapsed / 60:.2f} min")

assert returncode == 0

pred_files = sorted(PRED_DIR.glob("*.geff"))

print("\nPredicted GEFFs:", len(pred_files))
assert len(pred_files) == 40

print("FULL Fold 0 inference: OK")